# RA — Renewable AI Decision Engine (Notebook Demo)

This notebook is a **second front end** for the same RA engine that powers the FastAPI + React web app in `../backend` and `../frontend`. It is not a separate reimplementation — every number produced here comes from the exact same shared logic in [`ra_core/`](../ra_core): the synthetic data generator, the `GradientBoostingRegressor` forecaster, and the decision engine's scoring + explanation logic.

**Why this exists:** the web dashboard is the polished, judge-facing demo. This notebook is for exploring the engine directly — switching scenarios, running "what-if" conditions, and inspecting every intermediate number — without needing two servers running.

Run this top-to-bottom (`Run All`) and Section 2 will show the exact same current-state numbers as the web dashboard shows on a freshly-seeded `sunny` scenario (same seed, same default time step).


## 1. Setup

Locate and import `ra_core`, then generate the same deterministic synthetic dataset the web app uses (one full multi-day trace per scenario).

In [ ]:
import sys
from pathlib import Path


def _find_repo_root(start: Path, marker: str = "ra_core", max_up: int = 5) -> Path:
    p = start.resolve()
    for _ in range(max_up + 1):
        if (p / marker).is_dir():
            return p
        p = p.parent
    raise RuntimeError(
        f"Could not locate '{marker}/' above {start} -- launch Jupyter from the repo root "
        "(the folder containing ra_core/, backend/, frontend/, notebook/)."
    )


_repo_root = _find_repo_root(Path.cwd())
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

print(f"ra_core loaded from: {_repo_root / 'ra_core'}")


In [ ]:
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, HTML, Markdown, clear_output

from ra_core.config import (
    SCENARIOS, DEFAULT_SCENARIO, DEFAULT_START_INDEX, TOTAL_POINTS,
)
from ra_core.data_generator import generate_series
from ra_core.forecasting import forecast_surplus
from ra_core.decision_engine import evaluate
from ra_core import kpi

# One full deterministic trace per scenario -- identical to what the web
# app seeds into SQLite on first startup (same seed, same generator).
state = {
    "dfs": {s: generate_series(s) for s in SCENARIOS},
    "decision_log": [],       # notebook-session log ("Log this Decision" below)
    "last_result": None,
    "last_reading": None,
    "last_scenario": DEFAULT_SCENARIO,
    "last_index": DEFAULT_START_INDEX,
    "last_forecast": None,
}

print(f"Generated {len(SCENARIOS)} scenarios x {TOTAL_POINTS} timesteps "
      f"({TOTAL_POINTS * 15 / 60 / 24:.0f} days at 15-min resolution).")


### Display helpers

Small HTML/Plotly formatting helpers so the engine's output reads as cards and tables instead of raw dicts. These format numbers only -- they don't compute anything; every number they display comes straight out of `ra_core`.

In [ ]:
ACTION_LABELS = {
    "battery_charge": "Store in Battery",
    "water_pumping": "Run Water Pumping / Desalination",
    "sell_grid": "Sell to Grid",
    "curtail": "Curtail",
}
ACTION_COLORS = {
    "battery_charge": "#059669",
    "water_pumping": "#0891b2",
    "sell_grid": "#d97706",
    "curtail": "#64748b",
}


def _card(label, value, sub=""):
    sub_html = f'<div style="font-size:11px;color:#94a3b8;">{sub}</div>' if sub else ""
    return (
        '<div style="background:#f8fafc;border:1px solid #e2e8f0;border-radius:10px;'
        'padding:12px 16px;min-width:130px;">'
        f'<div style="font-size:11px;letter-spacing:.05em;text-transform:uppercase;color:#64748b;">{label}</div>'
        f'<div style="font-size:20px;font-weight:600;color:#0f172a;">{value}</div>{sub_html}</div>'
    )


def current_state_html(reading, surplus_kw):
    gen = reading["solar_kw"] + reading["wind_kw"]
    label = "Surplus" if surplus_kw >= 0 else "Deficit"
    color = "#059669" if surplus_kw >= 0 else "#dc2626"
    surplus_card = (
        f'<div style="background:#f8fafc;border:1px solid #e2e8f0;border-radius:10px;padding:12px 16px;min-width:130px;">'
        f'<div style="font-size:11px;letter-spacing:.05em;text-transform:uppercase;color:#64748b;">{label}</div>'
        f'<div style="font-size:20px;font-weight:600;color:{color};">{abs(surplus_kw):.1f} kW</div></div>'
    )
    cards = [
        _card("Generation", f"{gen:.1f} kW", f"Solar {reading['solar_kw']:.1f} · Wind {reading['wind_kw']:.1f}"),
        _card("Demand", f"{reading['demand_kw']:.1f} kW"),
        surplus_card,
        _card("Battery SoC", f"{reading['battery_soc']:.0f}%"),
        _card("Grid Price", f"{reading['price_egp']:.2f} EGP/kWh"),
        _card("Cloud Cover", f"{reading['cloud_cover'] * 100:.0f}%"),
        _card("Wind Speed", f"{reading['wind_speed']:.1f} m/s"),
        _card("Simulated Time", reading["timestamp"].replace("T", " ")[:16]),
    ]
    return f'<div style="display:flex;flex-wrap:wrap;gap:10px;margin:8px 0 16px;">{"".join(cards)}</div>'


def decision_card_html(rec):
    color = ACTION_COLORS.get(rec["action"], "#64748b")
    label = ACTION_LABELS.get(rec["action"], rec["action"])
    return f'''
    <div style="border:1px solid #e2e8f0;border-radius:12px;padding:16px 20px;background:#ffffff;">
      <div style="display:flex;justify-content:space-between;align-items:center;">
        <div style="font-weight:700;font-size:15px;color:#0f172a;">Recommended: {label}</div>
        <div style="background:{color};color:white;font-size:12px;padding:4px 10px;border-radius:999px;">{rec['action']}</div>
      </div>
      <div style="display:flex;gap:24px;margin:14px 0;flex-wrap:wrap;">
        <div><div style="font-size:18px;font-weight:600;">{rec['expected_kwh']:.1f} kWh</div><div style="font-size:11px;color:#64748b;">Energy</div></div>
        <div><div style="font-size:18px;font-weight:600;color:#059669;">{rec['expected_value_egp']:.1f} EGP</div><div style="font-size:11px;color:#64748b;">Expected value</div></div>
        <div><div style="font-size:18px;font-weight:600;color:#0284c7;">{rec['co2_avoided_kg']:.1f} kg</div><div style="font-size:11px;color:#64748b;">CO2 avoided</div></div>
        <div><div style="font-size:18px;font-weight:600;color:#7c3aed;">{rec['score']:.1f}</div><div style="font-size:11px;color:#64748b;">Decision score*</div></div>
      </div>
      <div style="background:#f8fafc;border:1px solid #e2e8f0;border-radius:8px;padding:12px 14px;font-size:13.5px;line-height:1.5;color:#334155;">{rec['explanation']}</div>
      <div style="font-size:10.5px;color:#94a3b8;margin-top:6px;">*Score = expected value + a small CO2 weighting used to rank actions. It is a transparent value ranking, not a probabilistic confidence.</div>
    </div>'''


def ranked_actions_html(ranked):
    rows = "".join(
        f'''<tr>
              <td style="padding:6px 10px;">{ACTION_LABELS.get(a['action'], a['action'])}</td>
              <td style="padding:6px 10px;text-align:right;">{a['expected_kwh']:.1f} kWh</td>
              <td style="padding:6px 10px;text-align:right;color:#059669;">{a['expected_value_egp']:.1f} EGP</td>
              <td style="padding:6px 10px;text-align:right;color:#0284c7;">{a['co2_avoided_kg']:.1f} kg</td>
              <td style="padding:6px 10px;text-align:right;font-weight:600;">{a['score']:.1f}</td>
            </tr>'''
        for a in ranked
    )
    return f'''
    <table style="border-collapse:collapse;width:100%;font-size:13.5px;">
      <thead><tr style="border-bottom:2px solid #e2e8f0;color:#64748b;text-align:right;">
        <th style="text-align:left;padding:6px 10px;">Action</th><th>Energy</th><th>Value</th><th>CO2 avoided</th><th>Score</th>
      </tr></thead>
      <tbody>{rows}</tbody>
    </table>'''


def build_forecast_figure(fc):
    history = fc["history"]
    future = fc["forecast"]
    h_times = [h["timestamp"] for h in history]
    h_actual = [h["actual_surplus_kw"] for h in history]
    f_times = [f["timestamp"] for f in future]
    f_forecast = [f["forecast_surplus_kw"] for f in future]
    f_actual_later = [f["actual_surplus_kw"] for f in future]

    now_time = h_times[-1] if h_times else (f_times[0] if f_times else None)
    bridge_time = [now_time] if now_time else []
    bridge_val = [h_actual[-1]] if h_actual else (f_forecast[:1] if f_forecast else [])

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=h_times, y=h_actual, name="Actual surplus",
                              mode="lines", line=dict(color="#0ea5e9", width=2)))
    fig.add_trace(go.Scatter(x=bridge_time + f_times, y=bridge_val + f_forecast,
                              name="Forecast surplus", mode="lines",
                              line=dict(color="#8b5cf6", width=2, dash="dash")))
    fig.add_trace(go.Scatter(x=f_times, y=f_actual_later, name="Actual (once known)",
                              mode="lines", line=dict(color="#0ea5e9", width=1), opacity=0.35))
    if now_time:
        # Use add_shape + add_annotation instead of the add_vline/add_hline
        # convenience wrappers -- those have known cross-version kwarg
        # forwarding issues (TypeError deep inside BaseFigure.add_vline)
        # depending on the installed plotly version. add_shape/add_annotation
        # are the stable, low-level primitives and avoid that failure mode.
        fig.add_shape(
            type="line", xref="x", yref="paper",
            x0=now_time, x1=now_time, y0=0, y1=1,
            line=dict(color="#f59e0b", width=1, dash="dot"),
        )
        fig.add_annotation(
            x=now_time, y=1.0, yref="paper", yanchor="bottom",
            text="now", showarrow=False, font=dict(color="#f59e0b", size=12),
        )
    fig.update_layout(
        height=340, margin=dict(l=40, r=20, t=30, b=40),
        yaxis_title="kW", template="plotly_white",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    )
    return fig


## 2–5. Live State → Forecast → Decision → Ranked Actions

These four views are wired together, exactly like the web dashboard: move the **Time step** slider (or click **+15 min** / **+1 hour**) to advance the simulated clock, or switch **Scenario**, and all four sections below update together. Click **Log this Decision** to append the current recommendation to the Section 8 history log.

In [ ]:
scenario_dd = widgets.Dropdown(options=SCENARIOS, value=DEFAULT_SCENARIO, description="Scenario:")
index_slider = widgets.IntSlider(value=DEFAULT_START_INDEX, min=0, max=TOTAL_POINTS - 1, step=1,
                                  description="Time step:", continuous_update=False,
                                  layout=widgets.Layout(width="520px"))
advance_15 = widgets.Button(description="+15 min")
advance_1h = widgets.Button(description="+1 hour")
log_button = widgets.Button(description="Log this Decision", button_style="primary")
log_feedback = widgets.HTML("")


def _on_scenario_change(change):
    if change["name"] == "value":
        index_slider.value = DEFAULT_START_INDEX


scenario_dd.observe(_on_scenario_change, names="value")
advance_15.on_click(lambda _: setattr(index_slider, "value", min(index_slider.value + 1, TOTAL_POINTS - 1)))
advance_1h.on_click(lambda _: setattr(index_slider, "value", min(index_slider.value + 4, TOTAL_POINTS - 1)))


def render_dashboard(scenario, idx):
    df = state["dfs"][scenario]
    reading = df.iloc[idx].to_dict()
    all_rows = df.to_dict("records")
    future_rows = df.iloc[idx + 1: idx + 13].to_dict("records")
    fc = forecast_surplus(all_rows, idx)
    future_prices = [r["price_egp"] for r in future_rows]
    surplus_kw = reading["solar_kw"] + reading["wind_kw"] - reading["demand_kw"]
    result = evaluate(reading, fc["forecast"], future_prices)

    state.update(last_result=result, last_reading=reading, last_scenario=scenario,
                 last_index=idx, last_forecast=fc)

    display(Markdown("#### 2. Current State"))
    display(HTML(current_state_html(reading, surplus_kw)))

    display(Markdown("#### 3. Forecast (next 6h)"))
    display(build_forecast_figure(fc))
    mq = fc["model_quality"]
    display(HTML(
        f'<div style="font-size:12px;color:#64748b;margin:-8px 0 12px;">'
        f'Model MAE (held-out 20% validation split) — generation {mq["generation_mae_kw"]} kW '
        f'· demand {mq["demand_mae_kw"]} kW</div>'
    ))

    display(Markdown("#### 4. AI Decision Engine"))
    display(HTML(decision_card_html(result["recommended"])))

    display(Markdown("#### 5. Priority Queue — Ranked Actions"))
    display(HTML(ranked_actions_html(result["ranked_actions"])))


def _on_log(_):
    if state["last_result"] is None:
        return
    rec = dict(state["last_result"]["recommended"])
    rec["timestamp"] = state["last_reading"]["timestamp"]
    rec["scenario"] = state["last_scenario"]
    state["decision_log"].append(rec)
    log_feedback.value = (
        f'<span style="color:#059669;font-size:12px;">Logged decision #{len(state["decision_log"])} '
        f'— {ACTION_LABELS.get(rec["action"], rec["action"])}</span>'
    )


log_button.on_click(_on_log)

controls = widgets.HBox([scenario_dd, index_slider, advance_15, advance_1h, log_button])
dashboard_view = widgets.interactive_output(render_dashboard, {"scenario": scenario_dd, "idx": index_slider})
display(controls, log_feedback, dashboard_view)


## 6. What-If Simulator

Explore hypothetical site conditions without altering the underlying scenario data. This reuses the exact same `forecast_surplus` and `evaluate` functions from `ra_core` — it only perturbs the *inputs* (solar capacity, demand growth, a transient dust-storm event), so any change you see here comes from the same shared decision engine, not a separate model.

- **Solar capacity** and **Demand growth** are treated as persistent site characteristics — they scale the entire trace (history + future).
- **Dust storm** is treated as a transient weather event — it only degrades solar output and forecast cloud cover for the upcoming forecast window, leaving history untouched.
- Sliders update the comparison live — no need to re-run the cell.

*Note: grid price is not re-simulated under these what-if conditions — it stays as originally generated for the scenario.*

In [ ]:
solar_mult_slider = widgets.FloatSlider(value=1.0, min=0.4, max=1.6, step=0.05, description="Solar capacity ×")
demand_mult_slider = widgets.FloatSlider(value=1.0, min=0.6, max=1.8, step=0.05, description="Demand ×")
dust_storm_toggle = widgets.Checkbox(value=False, description="Dust storm (next 6h)")


def run_whatif(solar_mult, demand_mult, dust_storm):
    if state["last_result"] is None:
        display(HTML('<div style="color:#94a3b8;">Run Section 2–5 above first.</div>'))
        return

    scenario = state["last_scenario"]
    idx = state["last_index"]
    base_df = state["dfs"][scenario]
    mod = base_df.copy()
    mod["solar_kw"] = (mod["solar_kw"] * solar_mult).clip(lower=0)
    mod["demand_kw"] = (mod["demand_kw"] * demand_mult).clip(lower=0.1)
    if dust_storm:
        end = min(idx + 24, len(mod) - 1)
        mod.loc[idx + 1: end, "solar_kw"] = (mod.loc[idx + 1: end, "solar_kw"] * 0.35).clip(lower=0)
        mod.loc[idx + 1: end, "cloud_cover"] = mod.loc[idx + 1: end, "cloud_cover"].clip(lower=0.85)

    reading = mod.iloc[idx].to_dict()
    all_rows = mod.to_dict("records")
    future_rows = mod.iloc[idx + 1: idx + 13].to_dict("records")
    fc = forecast_surplus(all_rows, idx)
    future_prices = [r["price_egp"] for r in future_rows]
    result = evaluate(reading, fc["forecast"], future_prices)

    base = state["last_result"]["recommended"]
    new = result["recommended"]

    tag = f"{solar_mult:.2f}× solar, {demand_mult:.2f}× demand" + (", dust storm" if dust_storm else "")
    display(Markdown("**Baseline vs. What-If recommendation**"))
    display(HTML(
        '<div style="display:flex;gap:16px;flex-wrap:wrap;">'
        f'<div style="flex:1;min-width:280px;"><div style="font-size:12px;color:#64748b;margin-bottom:4px;">'
        f'BASELINE (current dashboard state)</div>{decision_card_html(base)}</div>'
        f'<div style="flex:1;min-width:280px;"><div style="font-size:12px;color:#64748b;margin-bottom:4px;">'
        f'WHAT-IF ({tag})</div>{decision_card_html(new)}</div></div>'
    ))
    delta_value = new["expected_value_egp"] - base["expected_value_egp"]
    delta_co2 = new["co2_avoided_kg"] - base["co2_avoided_kg"]

    def sign(v):
        return f"+{v:.1f}" if v >= 0 else f"{v:.1f}"

    display(HTML(
        f'<div style="margin-top:10px;font-size:13px;color:#334155;">'
        f'Δ Expected value: <b>{sign(delta_value)} EGP</b> · Δ CO2 avoided: <b>{sign(delta_co2)} kg</b></div>'
    ))


whatif_controls = widgets.HBox([solar_mult_slider, demand_mult_slider, dust_storm_toggle])
whatif_view = widgets.interactive_output(
    run_whatif,
    {"solar_mult": solar_mult_slider, "demand_mult": demand_mult_slider, "dust_storm": dust_storm_toggle},
)
display(whatif_controls, whatif_view)


## 7. KPI Summary

The web dashboard doesn't currently expose an aggregate KPI view — it only shows per-decision numbers on the Decision Card and History timeline. This section aggregates those *same* per-decision fields (`expected_kwh`, `expected_value_egp`, `co2_avoided_kg`) across everything you've logged in this notebook session via **Log this Decision** above, so nothing here is a new or independently-derived number — it's a sum over numbers `ra_core.decision_engine` already produced.

In [ ]:
kpi_out = widgets.Output()
kpi_refresh = widgets.Button(description="Recalculate KPI Summary", button_style="info")


def render_kpi(_=None):
    scenario = state["last_scenario"]
    df = state["dfs"][scenario]
    all_rows = df.to_dict("records")
    available = kpi.total_available_surplus_kwh(all_rows, DEFAULT_START_INDEX, state["last_index"])
    session_decisions = [d for d in state["decision_log"] if d["scenario"] == scenario]
    summary = kpi.summarize(session_decisions, available)
    with kpi_out:
        clear_output(wait=True)
        cards = [
            _card("Decisions Logged", summary["decisions_logged"]),
            _card("Renewable Utilization", f'{summary["renewable_utilization_pct"]:.1f}%'),
            _card("Curtailment Avoided", f'{summary["curtailment_avoided_kwh"]:.1f} kWh'),
            _card("Total Value", f'{summary["total_value_egp"]:.1f} EGP'),
            _card("CO2 Avoided", f'{summary["total_co2_avoided_kg"]:.1f} kg'),
        ]
        display(HTML(f'<div style="display:flex;flex-wrap:wrap;gap:10px;">{"".join(cards)}</div>'))
        display(HTML(
            f'<div style="font-size:12px;color:#94a3b8;margin-top:6px;">Window: {scenario} scenario, '
            f'step {DEFAULT_START_INDEX} → {state["last_index"]} '
            f'({available:.1f} kWh of surplus was available in that window).</div>'
        ))


kpi_refresh.on_click(render_kpi)
display(kpi_refresh, kpi_out)
render_kpi()


## 8. History Log

Every decision you've clicked **Log this Decision** on, across scenarios, in this notebook session.

In [ ]:
history_out = widgets.Output()
history_refresh = widgets.Button(description="Refresh History Log")


def render_history(_=None):
    with history_out:
        clear_output(wait=True)
        if not state["decision_log"]:
            display(HTML(
                '<div style="color:#94a3b8;font-size:13px;">No decisions logged yet — '
                'use "Log this Decision" in Section 2–5 above.</div>'
            ))
            return
        rows = "".join(
            f'''<tr>
                  <td style="padding:6px 10px;">{d["timestamp"].replace("T", " ")[:16]}</td>
                  <td style="padding:6px 10px;">{d["scenario"]}</td>
                  <td style="padding:6px 10px;">{ACTION_LABELS.get(d["action"], d["action"])}</td>
                  <td style="padding:6px 10px;text-align:right;color:#059669;">{d["expected_value_egp"]:.1f} EGP</td>
                  <td style="padding:6px 10px;text-align:right;color:#0284c7;">{d["co2_avoided_kg"]:.1f} kg</td>
                </tr>'''
            for d in reversed(state["decision_log"])
        )
        display(HTML(
            '<table style="border-collapse:collapse;width:100%;font-size:13px;">'
            '<thead><tr style="border-bottom:2px solid #e2e8f0;color:#64748b;text-align:left;">'
            '<th style="padding:6px 10px;">Time</th><th>Scenario</th><th>Action</th>'
            '<th style="text-align:right;">Value</th><th style="text-align:right;">CO2</th>'
            f'</tr></thead><tbody>{rows}</tbody></table>'
        ))


history_refresh.on_click(render_history)
display(history_refresh, history_out)
render_history()


## Running as a standalone dashboard (optional)

Everything above works from **Run All** inside Jupyter/JupyterLab as-is. If you want a more polished, code-free presentation for a demo, this same notebook can be rendered as a standalone page with [Voilà](https://voila.readthedocs.io/):

```bash
pip install voila
voila RA_notebook_demo.ipynb
```

This hides all code cells and shows only the widgets/outputs. It's optional — the notebook works fine on its own.